# Замеры времени модели sage_fredt5_large на всех датасетах

In [ ]:
import time

import torch
import nest_asyncio
import pandas as pd
from tqdm import tqdm
from sage.evaluation import Scorer
from sage.spelling_correction import AvailableCorrectors
from sage.spelling_correction import T5ModelForSpellingCorruption
from sage.utils import load_available_dataset_from_hf, DatasetsAvailable

# В Jupyter уже работает event loop, поэтому нужен nest_asyncio
nest_asyncio.apply()

In [ ]:
# Конфигурация
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32

In [ ]:
# Модель и датасеты
MODEL_NAME = "sage_fredt5_large"
MODEL_PATH = AvailableCorrectors.sage_fredt5_large.value

In [ ]:
# Все русскоязычные датасеты из SAGE
RUSSIAN_DATASETS = {
    "RUSpellRU": DatasetsAvailable.RUSpellRU.name,
    "MultidomainGold": DatasetsAvailable.MultidomainGold.name,
    "MedSpellchecker": DatasetsAvailable.MedSpellchecker.name,
    "GitHubTypoCorpusRu": DatasetsAvailable.GitHubTypoCorpusRu.name,
}


# Функция для загрузки датасета
def load_dataset(dataset_name, split="test"):
    """Загружает датасет с ошибками"""
    print(f"   📥 Загрузка {dataset_name}...")
    sources, corrections = load_available_dataset_from_hf(
        dataset_name, for_labeler=True, split=split
    )

    print(f"   ✅ Загружено {len(sources)} примеров")
    return sources, corrections


# Функция для инференса с замером времени
def run_inference(model, texts, batch_size=32):
    """
    Запускает инференс и замеряет время
    Возвращает: (predictions, total_time, avg_time_per_sample, times_per_batch)
    """
    model.model.eval()
    predictions = []
    total_time = 0.0
    batch_times = []

    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc="      Инференс"):
            batch_texts = texts[i : i + batch_size]

            # Замер времени на батч
            start_time = time.time()

            # Инференс для каждого текста в батче
            batch_predictions = []
            for text in batch_texts:
                pred = model.correct(text)
                batch_predictions.append(pred[0] if pred else text)

            batch_time = time.time() - start_time
            total_time += batch_time
            batch_times.append(batch_time)

            predictions.extend(batch_predictions)

    avg_time_per_sample = total_time / len(texts) if texts else 0
    return predictions, total_time, avg_time_per_sample, batch_times


# Функция для оценки качества (опционально)
def calculate_metrics(predictions, references, metrics=["ruspelleval"]):
    """Рассчитывает метрики качества"""
    scorer = Scorer()
    # Используем predictions как sources, а references как corrections
    metrics_dict = scorer.score(predictions, references, references, metrics=metrics)
    return metrics_dict

In [ ]:
# Загружаем модель
model = T5ModelForSpellingCorruption.from_pretrained(MODEL_PATH)
model.model.to(DEVICE)

In [ ]:
# Словари для хранения результатов
all_results = []
all_predictions = {}
all_metrics = {}

# Прогоняем модель на каждом датасете
for dataset_name, dataset_path in RUSSIAN_DATASETS.items():
    print(f"\n{'=' * 60}")
    print(f"📂 ТЕСТИРОВАНИЕ НА {dataset_name}")
    print(f"{'=' * 60}")

    # Загружаем данные
    try:
        sources, references = load_dataset(dataset_path, split="test")
        if not sources:
            print(f"   ⚠️ Нет данных для {dataset_name}")
            continue
    except Exception as e:
        print(f"   ❌ Ошибка загрузки: {e}")
        continue

    # Замеряем инференс
    print("   🚀 Запуск инференса...")
    predictions, inference_time, avg_time, batch_times = run_inference(
        model, sources, batch_size=BATCH_SIZE
    )

    # Рассчитываем метрики качества (опционально)
    print("   📊 Расчет метрик качества...")
    try:
        metrics = calculate_metrics(predictions, references, metrics=["ruspelleval"])
        f1_spell = metrics.get("SPELL_F1", metrics.get("F1", 0))
        precision_spell = metrics.get("SPELL_Precision", metrics.get("Precision", 0))
        recall_spell = metrics.get("SPELL_Recall", metrics.get("Recall", 0))
    except Exception as e:
        print(f"   ⚠️ Не удалось рассчитать метрики: {e}")
        f1_spell = precision_spell = recall_spell = 0

    # Сохраняем результаты
    result = {
        "Dataset": dataset_name,
        "Num_Samples": len(sources),
        "Batch_Size": BATCH_SIZE,
        "Total_Inference_Time_sec": round(inference_time, 3),
        "Avg_Time_Per_Sample_ms": round(avg_time * 1000, 3),
        "Avg_Time_Per_Batch_sec": round(sum(batch_times) / len(batch_times), 3)
        if batch_times
        else 0,
        "F1_SPELL": round(f1_spell, 2),
        "Precision_SPELL": round(precision_spell, 2),
        "Recall_SPELL": round(recall_spell, 2),
        "Samples_Per_Second": round(len(sources) / inference_time, 2),
    }
    all_results.append(result)

    # Сохраняем предсказания
    pred_df = pd.DataFrame(
        {
            "original_text": sources,
            "corrected_text": predictions,
            "reference_text": references,
        }
    )
    all_predictions[dataset_name] = pred_df
    all_metrics[dataset_name] = metrics

    # Выводим результаты
    print(f"\n   ⏱️  РЕЗУЛЬТАТЫ ДЛЯ {dataset_name}:")
    print(f"      • Общее время инференса: {inference_time:.3f} сек")
    print(f"      • Среднее время на пример: {avg_time * 1000:.3f} мс")
    print(f"      • Примеров в секунду: {len(sources) / inference_time:.2f}")
    print(f"      • F1 (spell): {f1_spell:.2f}")


📂 ТЕСТИРОВАНИЕ НА RUSpellRU
   📥 Загрузка RUSpellRU...
   ✅ Загружено 2008 примеров
   🚀 Запуск инференса...


      Инференс: 100%|██████████| 63/63 [04:43<00:00,  4.51s/it]


   📊 Расчет метрик качества...


Calculating words metric: 100%|██████████| 2008/2008 [00:01<00:00, 1442.06it/s]



   ⏱️  РЕЗУЛЬТАТЫ ДЛЯ RUSpellRU:
      • Общее время инференса: 283.924 сек
      • Среднее время на пример: 141.396 мс
      • Примеров в секунду: 7.07
      • F1 (spell): 100.00

📂 ТЕСТИРОВАНИЕ НА MultidomainGold
   📥 Загрузка MultidomainGold...
   ✅ Загружено 4106 примеров
   🚀 Запуск инференса...


      Инференс: 100%|██████████| 129/129 [20:43<00:00,  9.64s/it]


   📊 Расчет метрик качества...


Calculating words metric:  86%|████████▋ | 3542/4106 [00:28<00:06, 87.92it/s]  /home/robot/projects/sage/sage/evaluation/ruspelleval.py:359: UserWarning: Skipping 3566 line, because operation timed out...
  warnings.warn("Skipping {} line, because operation timed out...".format(num), UserWarning)
Calculating words metric:  87%|████████▋ | 3568/4106 [00:30<00:21, 24.48it/s]/home/robot/projects/sage/sage/evaluation/ruspelleval.py:359: UserWarning: Skipping 3570 line, because operation timed out...
  warnings.warn("Skipping {} line, because operation timed out...".format(num), UserWarning)
Calculating words metric:  87%|████████▋ | 3568/4106 [00:44<00:21, 24.48it/s]/home/robot/projects/sage/sage/evaluation/ruspelleval.py:359: UserWarning: Skipping 3580 line, because operation timed out...
  warnings.warn("Skipping {} line, because operation timed out...".format(num), UserWarning)
Calculating words metric:  87%|████████▋ | 3585/4106 [01:04<01:01,  8.48it/s]/home/robot/projects/sage/sage/ev

В результате получили следующие результаты:
- Датасет: RUSpellRU
- Примеров: 2008
- ⏱️ Время выполнения инференса: 283.924 секунд
- 📊 Среднее время на пример: 0.141 секунд